# Swiggy Restaurant Q&A Chatbot (RAG-style with ChatXAI)

This notebook creates an interactive Q&A chatbot that answers questions about the cleaned Swiggy dataset.

**Approach**:
- Load cleaned CSV
- Extract each restaurant as a text chunk (knowledge base)
- Simple keyword-based retrieval (lightweight, no heavy embedding models)
- Use **ChatXAI (grok-4-latest)** to answer based on retrieved chunks
- Interactive `input()` loop

**Requirements**: Set your `XAI_API_KEY` environment variable before running.


In [ ]:
import os
import pandas as pd
import re
from langchain_xai import ChatXAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Check API key
if not os.getenv("XAI_API_KEY"):
    print("WARNING: XAI_API_KEY not found. Please set it:")
    print("  os.environ['XAI_API_KEY'] = 'your-key-here'")
else:
    print("XAI_API_KEY detected.")


In [ ]:
# Load the cleaned dataset
df = pd.read_csv('/workspace/attachments/swiggy_cleaned-2.csv')
print(f"Loaded {len(df)} restaurants")
print("Columns:", df.columns.tolist())
print("\nSample:")
display(df.head(3))


## Step 1: Extract & Chunk the Knowledge Base

Each restaurant row becomes a natural language chunk.

In [ ]:
def row_to_chunk(row):
    """Convert a restaurant row into a readable text chunk."""
    rating = row['Rating'] if pd.notna(row['Rating']) else "Not available"
    return (
        f"Restaurant: {row['Shop_Name']}. "
        f"Cuisines: {row['Cuisine']}. "
        f"Location: {row['Location']}. "
        f"Area: {row['Area']}. "
        f"Rating: {rating}. "
        f"Cost for Two: Rs.{row['Cost_for_Two']}. "
        f"Cost Bucket: {row['Cost_Bucket']}."
    )

# Create knowledge base chunks
chunks = [row_to_chunk(row) for _, row in df.iterrows()]
print(f"Created {len(chunks)} chunks")
print("\nExample chunk:")
print(chunks[0])


## Step 2: Simple Retrieval Function

Keyword-based retrieval (lightweight alternative to embeddings).

In [ ]:
def retrieve_chunks(query: str, chunks: list, top_k: int = 8):
    """Simple keyword matching retrieval."""
    query_lower = query.lower()
    query_words = set(re.findall(r'\w+', query_lower))
    
    scored = []
    for chunk in chunks:
        chunk_lower = chunk.lower()
        # Score by number of matching words
        matches = sum(1 for w in query_words if w in chunk_lower)
        if matches > 0:
            scored.append((matches, chunk))
    
    # Sort by score descending
    scored.sort(key=lambda x: x[0], reverse=True)
    top_chunks = [chunk for _, chunk in scored[:top_k]]
    
    # Fallback: if nothing matched, return a sample of all
    if not top_chunks:
        top_chunks = chunks[:top_k]
    
    return top_chunks

# Test retrieval
test_q = "best biryani in koramangala under 400"
print("Test retrieval for:", test_q)
for c in retrieve_chunks(test_q, chunks, top_k=3):
    print("-", c[:120] + "...")


## Step 3: Initialize ChatXAI (Grok)

In [ ]:
# Initialize the model
llm = ChatXAI(
    model="grok-4-latest",
    temperature=0.1,
    max_tokens=1024
)

print("ChatXAI (grok-4-latest) initialized.")


## Step 4: Build the QA Chain

In [ ]:
# Improved Prompt Template for complex questions
system_msg = """You are an expert data analyst assistant specializing in restaurant data from Swiggy (Bangalore areas: Koramangala, HSR, BTM, Jayanagar).

You answer questions using ONLY the restaurant data provided in the context. Do not invent data.

## How to Handle Questions

**For simple lookups** (specific restaurant, rating, cost):
- Answer directly with the facts.

**For complex / analytical questions** (comparisons, rankings, aggregations, filters, "best", "average", "how many", "which area", multi-condition):
1. Carefully analyze the relevant restaurant records in the context.
2. Perform the required filtering, counting, sorting, or calculation mentally.
3. Present a clear, structured answer.
4. When listing restaurants, use a clean format (bullet or numbered list) showing: Name | Area | Rating | Cost for Two.
5. If you calculate averages, counts, or comparisons, briefly show your reasoning or the key numbers.

**Rules**:
- Prefer precise numbers over vague statements.
- If the data is insufficient to fully answer, say what you can answer and what is missing.
- Use "Rs." for costs.
- Keep answers concise but complete.
- If multiple restaurants match, list the top relevant ones (usually up to 8–10).
- For "best value" or similar: consider both high rating and lower cost.
- Never make up restaurants, ratings, or costs that are not in the provided data.

Always base your answer strictly on the restaurant records given."""

human_msg = """Restaurant Records:
{context}

Question: {question}

Think step by step if the question is complex, then provide the final answer:"""

prompt = ChatPromptTemplate.from_messages([
    ("system", system_msg),
    ("human", human_msg)
])

chain = prompt | llm | StrOutputParser()
print("Improved QA chain ready (handles complex questions).")


## Step 5: Interactive Q&A Chatbot

Run the cell below and start asking questions. Type `exit` or `quit` to stop.

In [ ]:
def ask_question(question: str):
    """Retrieve relevant chunks and generate answer."""
    relevant_chunks = retrieve_chunks(question, chunks, top_k=10)
    context = "\n".join(relevant_chunks)
    
    response = chain.invoke({
        "context": context,
        "question": question
    })
    return response

# Interactive loop
print("=" * 60)
print("Swiggy Restaurant Q&A Chatbot")
print("Ask anything about the restaurants (areas, ratings, costs, cuisines, etc.)")
print("Type 'exit' or 'quit' to stop.")
print("=" * 60)

while True:
    user_query = input("\nYou: ").strip()
    if user_query.lower() in ["exit", "quit", "q"]:
        print("Goodbye!")
        break
    if not user_query:
        continue
    
    print("\nThinking...")
    try:
        answer = ask_question(user_query)
        print(f"\nBot: {answer}")
    except Exception as e:
        print(f"Error: {e}")
        print("Make sure XAI_API_KEY is set correctly.")


## Example Questions You Can Ask

- What are the highest rated restaurants in Koramangala?
- Show me biryani places under Rs.400
- Which area has the most restaurants?
- What is the average cost for two in HSR?
- List ice cream shops with rating above 4.5
- Best value North Indian restaurants
- How many restaurants have rating 4.3 or higher?
- Compare costs between BTM and Koramangala
